# Frozen speech features: one-recording check

Özgür Ekin Sönmez

Measure the fixed wav2vec 2.0 encoder on one public train recording before extracting the pilot. No labels, development audio, fine-tuning or model comparison is performed here. Use a free GPU runtime; do not upgrade to a paid plan. The uploaded bundle reuses verified local weights, so no second checkpoint download is needed.

This v2 notebook is a clean recipe for a fresh runtime. It resolves encoder packages and installs them in an isolated environment, reusing runtime GPU PyTorch dependency directories. It does not install encoder packages globally.

The earlier live session used v1 and encountered global package conflicts. Its first isolation probe also expected an incorrect matrix sum (16 instead of 8). Those attempts remain separate historical evidence. This clean recipe has not reproduced that session.


In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path
import torch
hardware = {'python': sys.version, 'torch': torch.__version__, 'cuda': torch.cuda.is_available(),
            'disk_free_bytes': shutil.disk_usage('/content').free,
            'memory': Path('/proc/meminfo').read_text()}
if torch.cuda.is_available():
    hardware['gpu_name'] = torch.cuda.get_device_name(0)
    hardware['gpu_free_total_bytes'] = list(torch.cuda.mem_get_info())
Path('/content/ssl_hardware.json').write_text(json.dumps(hardware, indent=2))
print(json.dumps(hardware, indent=2))
assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > T4 GPU, then reconnect.'
assert tuple(map(int, torch.__version__.split('+')[0].split('.')[:2])) >= (2,10), 'Patched PyTorch >=2.10 is required. Stop and review this runtime; do not install an unplanned GPU stack.'
assert hardware['disk_free_bytes'] >= 20 * 1024**3

## Resolve and lock the optional packages

Keep the runtime-provided patched GPU PyTorch. Resolve and hash-lock encoder packages before installing them into a separate environment. This step may fetch wheels into the pip cache but does not install them globally. Review the wheel list and transfer budget before proceeding.


In [ ]:
import hashlib, importlib.metadata as metadata
pins = ['transformers==4.57.6','numpy==2.2.6','soundfile==0.13.1',
        'huggingface-hub==0.36.2','tokenizers==0.22.2','safetensors==0.8.0']
subprocess.run([sys.executable,'-m','pip','install','--dry-run','--ignore-installed',
                '--only-binary=:all:','--report','/content/ssl_resolve.json',*pins], check=True)
resolution = json.loads(Path('/content/ssl_resolve.json').read_text())
assert not any(item['metadata']['name'].lower() == 'torch' for item in resolution['install'])
lock = []
for item in resolution['install']:
    download = item['download_info']
    lock.append(download['url']+' --hash=sha256:'+download['archive_info']['hashes']['sha256'])
Path('/content/ssl-linux.lock').write_text('\n'.join(lock)+'\n')
print('Dependency lock saved. No encoder packages were installed globally.')


## Prepare the isolated GPU environment

The corrected helper creates a fresh venv without global site-packages access. It reuses runtime Torch dependency directories and distribution metadata, then installs the hash-locked encoder packages. It records shared paths, metadata hashes, pip check and a CUDA import probe. Shared namespace directories can contain additional runtime files; this is environment isolation, not a security sandbox. Stop on failure and preserve logs.


In [ ]:
isolation_source = '"""Create an isolated Colab encoder environment, reusing runtime GPU wheels.\n\nDoes not change the encoder, install another torch build or modify global packages.\nRun after preserving the original dependency resolution and failed pip check.\n"""\nimport hashlib\nimport importlib.metadata as metadata\nimport json\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport venv\n\nfrom packaging.requirements import Requirement\nfrom packaging.utils import canonicalize_name\n\n\ndef main():\n    root = Path(\'/content/ssl-isolated-v1\')\n    assert not root.exists(), \'Preserve previous environment attempts\'\n    resolution = json.loads(Path(\'/content/ssl_resolve.json\').read_text())\n    owned = {canonicalize_name(i[\'metadata\'][\'name\']) for i in resolution[\'install\']}\n    assert \'torch\' not in owned\n    # Only the runtime torch dependency closure may enter the isolated environment.\n    pending, reused = [\'torch\'], {}\n    while pending:\n        name = canonicalize_name(pending.pop())\n        if name in owned or name in reused:\n            continue\n        dist = metadata.distribution(name)\n        reused[name] = dist\n        for text in dist.requires or []:\n            requirement = Requirement(text)\n            if requirement.marker is None or requirement.marker.evaluate({\'extra\': \'\'}):\n                pending.append(requirement.name)\n    venv.EnvBuilder(with_pip=False, system_site_packages=False).create(root)\n    python = root / \'bin/python\'\n    site = next((root / \'lib\').glob(\'python*/site-packages\'))\n    links = {}\n    provenance = {}\n    for name, dist in sorted(reused.items()):\n        provenance[name] = {\'version\': dist.version, \'root\': str(dist.locate_file(\'\')),\n                            \'metadata_hashes\': {}}\n        for file in dist.files or []:\n            if not file.parts or file.parts[0] in (\'..\', \'.\'):\n                continue\n            source = Path(dist.locate_file(file.parts[0])).resolve()\n            if file.name in (\'RECORD\', \'METADATA\') and \'.dist-info\' in str(file):\n                provenance[name][\'metadata_hashes\'][str(file)] = hashlib.sha256(\n                    Path(dist.locate_file(file)).read_bytes()).hexdigest()\n            target = site / file.parts[0]\n            if target.is_symlink():\n                assert target.resolve() == source, \'Conflicting reused package paths\'\n            elif target.exists():\n                raise RuntimeError(f\'Unexpected existing environment path: {target}\')\n            else:\n                target.symlink_to(source, target_is_directory=source.is_dir())\n            links[str(target.relative_to(root))] = str(source)\n    receipt = {\'runtime_torch_packages\': {k: d.version for k, d in reused.items()},\n               \'provenance\': provenance,\n               \'shared_paths\': links, \'system_site_packages\': False,\n               \'lock_sha256\': hashlib.sha256(Path(\'/content/ssl-linux.lock\').read_bytes()).hexdigest()}\n    Path(\'/content/ssl_isolation.json\').write_text(json.dumps(receipt, indent=2))\n    subprocess.run([sys.executable, \'-m\', \'pip\', \'--python\', str(python), \'install\',\n                    \'--no-deps\', \'--require-hashes\', \'-r\', \'/content/ssl-linux.lock\'], check=True)\n    check = subprocess.run([sys.executable, \'-m\', \'pip\', \'--python\', str(python), \'check\'],\n                           capture_output=True, text=True)\n    Path(\'/content/ssl_isolated_pip_check.txt\').write_text(check.stdout + check.stderr)\n    print(check.stdout + check.stderr, flush=True)\n    check.check_returncode()\n    probe = """import json, importlib.metadata as m, torch\nfrom transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor\nimport numpy, soundfile\nassert torch.cuda.is_available()\nx = torch.ones((2, 2), device=\'cuda\')\nassert (x @ x).sum().item() == 8\nprint(json.dumps({\'versions\': {d.metadata[\'Name\']: d.version for d in m.distributions()},\n                  \'gpu\': torch.cuda.get_device_name(0), \'cuda_matmul\': \'passed\'}, indent=2))\n"""\n    result = subprocess.run([str(python), \'-c\', probe], capture_output=True, text=True, timeout=90)\n    Path(\'/content/ssl_isolated_probe.txt\').write_text(result.stdout + result.stderr)\n    print(result.stdout + result.stderr, flush=True)\n    result.check_returncode()\n    Path(\'/content/ssl_isolated_environment.json\').write_text(result.stdout)\n    print(\'Isolated encoder environment passed. Use /content/ssl-isolated-v1/bin/python for the worker.\')\n\n\nif __name__ == \'__main__\':\n    main()\n'
Path('/content/prepare_colab_ssl_env.py').write_text(isolation_source, encoding='utf-8')
subprocess.run([sys.executable, '/content/prepare_colab_ssl_env.py'], check=True)


## Upload the local smoke bundle

Select `ssl_colab_smoke_v1.zip` from the project outputs folder. It contains the fixed public training recording, verified model files, geometry code, the measured CPU feature reference and the bounded smoke runner. No private recordings, labels or development audio are included.


In [ ]:
from google.colab import files
uploaded = files.upload()
assert len(uploaded) == 1 and 'ssl_colab_smoke_v1.zip' in uploaded
import zipfile
bundle = Path('/content/ssl-smoke-bundle')
assert not bundle.exists(), 'Keep previous attempts. Use a fresh runtime for another smoke.'
bundle.mkdir()
with zipfile.ZipFile('/content/ssl_colab_smoke_v1.zip') as archive:
    assert sum(item.file_size for item in archive.infolist()) < 500_000_000
    for item in archive.infolist():
        target = (bundle / item.filename).resolve()
        assert target.is_relative_to(bundle.resolve()), 'Unsafe archive path'
    archive.extractall(bundle)
print('Bundle extracted. The worker will verify every pinned payload before loading.')

## Run the bounded GPU check

The parent watches system memory and the worker process for 180 seconds. The worker records GPU memory and compares GPU features with the preliminary CPU reference; numerical differences are reported, not tuned away. A stopped attempt is kept and must be reviewed. This notebook does not launch the 192-recording extraction automatically.


In [ ]:
command = ['/content/ssl-isolated-v1/bin/python', str(bundle/'ssl_colab_smoke.py'),'--execute',
           '--bundle-dir', str(bundle),'--output-dir','/content/ssl-colab-smoke-v1']
completed = subprocess.run(command, timeout=200)
assert completed.returncode == 0, 'Smoke stopped: inspect status and worker logs; do not automatically retry.'
print(Path('/content/ssl-colab-smoke-v1/status.json').read_text())
print(Path('/content/ssl-colab-smoke-v1/result.json').read_text())

In [ ]:
import zipfile
result_zip = Path('/content/ssl_colab_smoke_result_v2.zip')
with zipfile.ZipFile(result_zip,'x',compression=zipfile.ZIP_DEFLATED) as archive:
    for path in Path('/content/ssl-colab-smoke-v1').rglob('*'):
        if path.is_file(): archive.write(path, str(path.relative_to('/content')))
    for name in ['ssl_hardware.json','ssl_resolve.json','ssl-linux.lock','ssl_isolation.json','ssl_isolated_pip_check.txt','ssl_isolated_probe.txt',
                 'ssl_isolated_environment.json','prepare_colab_ssl_env.py']:
        archive.write('/content/'+name,name)
files.download(str(result_zip))